## 📑 Core ML/ SB Government Guaranteed Bucket 7+
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AYCFHQ)**  
  Comprehensive documentation outlining the business requirements for this report.

#####Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, regexp_replace, array, col, explode, lit, when, split
import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim
import boto3
import os

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

ReferenceDateUnion = spark.sql(f'''
                    SELECT last_day(add_months('{run_date}', -1)) AS SOM
                         ''').collect()[0]['SOM']

Month = spark.sql(f'''
                    SELECT date_format('{run_date}','yyyyMM') AS Months
                         ''').collect()[0]['Months']

##### Schema of the table

In [0]:
table_name = 'Core_MLSBGovernmentGuaranteedBucket7Plus' 

schema = (
"BusinessDate date," 
"AssetSegment string,"
"ExposureKey string,"
"AgreementKey string,"
"CDI string,"
"PartyTaxRegistrationNumber string,"
"NplThr string,"
"N3869 string,"
"Product string,"
"SUBPRODUCT string,"
"FinalDEKStatus string,"
"Enstasi boolean,"
"RtlWhl string,"
"QueDesc string,"
"CASE_DENOUNCE_STATUS string,"
"Agency string,"
"CaseBucket int,"
"CASE_DEBT_AMT decimal(19,2),"
"CASE_OUTSTANDING_AMT decimal(19,2),"
"ExceptionReason string,"
"InsertTimeStamp timestamp"
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date,  # type: ignore
)


##### Create Delta Tables with selected fields & Temporary Views for each delta table

In [0]:
# file_path = f"/Volumes/{env_prefix}-cdh/reference/repoint/Covid19.csv" 
# Covid19Static = spark.read.csv(file_path, header=True, inferSchema=False, sep=";")
# Covid19Static.createOrReplaceTempView("Covid19Static")

In [0]:
Covid19Static = spark.sql(
f'''
select * from `{env_prefix}-reportinputfiles`.repoint.core_staticcovid19
'''
)
Covid19Static.createOrReplaceTempView("Covid19Static")

In [0]:
# file_path = f"/Volumes/qa-cdh/reference/repoint/ExceptionQueues.csv"
# ExceptionQueuesStatic = spark.read.csv(file_path, header=True, inferSchema=False, sep=";")
# ExceptionQueuesStatic.createOrReplaceTempView("ExceptionQueuesStatic")

In [0]:
ExceptionQueuesStatic = spark.sql(
f'''
select * from `{env_prefix}-reportinputfiles`.repoint.core_staticexceptionqueues
'''
)
ExceptionQueuesStatic.createOrReplaceTempView("ExceptionQueuesStatic")

In [0]:
# file_path = f"/Volumes/qa-cdh/reference/repoint/ExceptionComments.csv"
# ExceptionCommentsStatic = spark.read.csv(file_path, header=True, inferSchema=False, sep=";")
# ExceptionCommentsStatic.createOrReplaceTempView("ExceptionCommentsStatic")

In [0]:
ExceptionCommentsStatic = spark.sql(
f'''
select * from `{env_prefix}-reportinputfiles`.repoint.core_staticexceptioncomments
'''
)
ExceptionCommentsStatic.createOrReplaceTempView("ExceptionCommentsStatic")

In [0]:
# file_path = f"/Volumes/{env_prefix}-cdh/reference/repoint/Apografi.csv"
# ApografiStatic = spark.read.csv(file_path, header=True, inferSchema=False, sep=";")
# ApografiStatic.createOrReplaceTempView("ApografiStatic")

### Tables from Databricks as Temps

In [0]:
Covid19 = spark.sql(
f'''
    SELECT *
    FROM Covid19Static
    WHERE ReferenceDate = '{run_date}'
'''
)
Covid19.createOrReplaceTempView("Covid19")

In [0]:
ExceptionQueues = spark.sql(
f'''
    SELECT *
    FROM ExceptionQueuesStatic
    WHERE ReferenceDate = (SELECT MAX(ReferenceDate) FROM ExceptionQueuesStatic WHERE ReferenceDate <= '{run_date}')
'''
)
ExceptionQueues.createOrReplaceTempView("ExceptionQueues")

In [0]:
ExceptionComments = spark.sql(
f'''
    SELECT *
    FROM ExceptionCommentsStatic
    WHERE ReferenceDate = (SELECT MAX(ReferenceDate) FROM ExceptionCommentsStatic WHERE ReferenceDate <= '{run_date}')
'''
)
ExceptionComments.createOrReplaceTempView("ExceptionComments")

In [0]:
ArxeioApografis_1 = spark.sql(
f'''
    SELECT *
    FROM (SELECT DISTINCT ExposureKey, ExposureKeyWithConfirmedDebts, Comments, BusinessDate FROM `{env_prefix}-cdh`.externalfiles.extinventory) a
    WHERE BusinessDate = (SELECT MAX(businessdate) FROM `{env_prefix}-cdh`.externalfiles.extinventory WHERE BusinessDate <= '{run_date}')

    -- SELECT *
    -- FROM ApografiStatic 
	-- WHERE ReferenceDate = (SELECT MAX(ReferenceDate) FROM ApografiStatic WHERE ReferenceDate <= '{run_date}') --20250901 FM Change from static file to table
'''
)
ArxeioApografis_1.createOrReplaceTempView("ArxeioApografis_1")

In [0]:
# Step 1: Create a temporary view from the SQL temporary table
df_arxeio_apografis_1 = spark.sql("SELECT * FROM ArxeioApografis_1")
 
# Step 2: Create a new DataFrame with the xmlfilter column
df_f1 = df_arxeio_apografis_1.selectExpr(
    "*",
    "concat('<X>', regexp_replace(ExposureKey, '-', '</X><X>'), '</X>') as xmlfilter"
)
# df_f1 = df_arxeio_apografis_1.selectExpr(
#     "*",
#     "concat('<X>', regexp_replace(`Αριθμός λογαριασμού`, '-', '</X><X>'), '</X>') as xmlfilter"
# ) --20250901 FM
 
# Step 3: Explode the xmlfilter into separate rows
df_exploded = df_f1.select("*", explode(split(col("xmlfilter"), "</X><X>")).alias("AccountNumber"))
 
# Step 4: Replace '<X>' with empty string
df_exploded = df_exploded.withColumn("AccountNumber", regexp_replace(col("AccountNumber"),'<X>',''))
df_exploded = df_exploded.withColumn("AccountNumber", regexp_replace(col("AccountNumber"),'</X>',''))

# Step 5: Create a temporary view for the result if needed
df_exploded.createOrReplaceTempView("ArxeioApografis")

# Step 6: Select the final columns
df_result = df_exploded.select("*")  

# Show the result
# df_result.show()


In [0]:
UnionFile = spark.sql(
f'''
    SELECT	*
    FROM `{env_prefix}-cdh`.externalfiles.extunionfile 
    WHERE SnapshotDate = '{ReferenceDateUnion}' 
    AND cast(IngestionTimeStamp as date) = (Select Max(cast(IngestionTimeStamp as date)) from `{env_prefix}-cdh`.externalfiles.extunionfile where SnapshotDate = '{ReferenceDateUnion}')
'''
)
UnionFile.createOrReplaceTempView("UnionFile")

# Check if the dataset is empty and print a message accordingly
if UnionFile.count() == 0:
    print(f"The required external file extunionfile dataset for date {ReferenceDateUnion!r} is empty")

In [0]:
Account = spark.sql(
f'''
    SELECT a.AssetSegment
          ,a.ExposureKey
          ,a.AgreementKey
          ,a.CDI
          ,a.PartyTaxRegistrationNumber
          ,a.Exposurestatus
          ,a.ExposurestatusCode
          ,a.DueDebtAmount
          ,a.DueAmount
          ,a.OutstandingTotalBalance
          ,a.OnBalance
          ,a.InterestRate
          ,a.Bucket
          ,a.PartyTaxRegistrationNumber
          ,l.AgencyCode
          ,l.Agency
          ,a.BusinessDate
          ,a.CaseCode
          ,a.ProductKey
          ,a.SourceSystem
          ,a.PartyRelationshipType
          ,a.PortfolioName
          ,a.IsActiveFlag
          ,a.Currency
          ,a.IsGuaranteeFlag
    FROM `{env_prefix}-cdh`.primary.exposure             a
    LEFT JOIN UnionFile u
    ON u.ExposureKey = a.ExposureKey
    LEFT JOIN `{env_prefix}-cdh`.primary.casecode l
    ON l.CasecodeKey = a.Casecode AND l.BusinessDate = a.BusinessDate AND l.PortfolioName = 'Core Retail' 
    WHERE a.IsActiveFlag = TRUE AND a.BusinessDate = '{run_date}' 
    AND a.PortfolioName = 'Core Retail' AND a.PartyRelationshipType = 'Primary Owner'
  
'''
)
Account.createOrReplaceTempView("Account")

In [0]:
Underlaw = spark.sql(
f'''
SELECT * FROM (
        SELECT  CDI
          , InLawFlag
          , LawStatus
          , CalcLawStatus
           , rank() over (partition by CDI order by BusinessDate desc) as rank
        FROM `{env_prefix}-cdh`.primary.UnderLaw
        WHERE BusinessDate <= '{run_date}'
        AND PortfolioName = 'Core Retail'
        AND UnderLawType = 'N3869'
        AND sourcesystem = 'qca' 
        )
    WHERE RANK = 1
    AND InLawFlag = 1
'''
)
Underlaw.createOrReplaceTempView("Underlaw")

In [0]:
 DenouncementPool = spark.sql(
 f'''
     SELECT ExposureKey
        , ContractCoCFinal	
      	, AppealExclusionContract
 	from `p-ptools`.perimeters.legalcore_v3
 	WHERE BusinessDate  = (SELECT MAX(BusinessDate) FROM `p-ptools`.perimeters.legalcore_v3 WHERE BusinessDate <='{run_date}' )
    AND _creation_timestamp = (SELECT MAX(_creation_timestamp) FROM `p-ptools`.perimeters.legalcore_v3 WHERE BusinessDate = (SELECT MAX(BusinessDate) FROM `p-ptools`.perimeters.legalcore_v3 WHERE BusinessDate <='{run_date}' ) )
    and PartyRelationshipType = 'Primary Owner'
  '''
 )
 DenouncementPool.createOrReplaceTempView("DenouncementPool")


In [0]:
MLPerimeter = spark.sql(
f'''
		SELECT ac.AssetSegment
		     , ac.ExposureKey      
			 , ac.AgreementKey
			 , ac.CDI
			 , ac.PartyTaxRegistrationNumber
			 , CASE WHEN u.BucketCategoryWithMateriality <= 3 THEN 'THR' 
			        WHEN u.BucketCategoryWithMateriality > 3  THEN 'NPL'
					END                                                              AS NplThr
             , CASE WHEN l.CDI = ac.CDI THEN 'N3869' 
			        ELSE 'No' END                                                    AS N3869
			 , p.SubProduct 														 AS Product                                                             
			 , CONCAT(p.BankingUnitCode,' - ',p.ProductDescription) 				 AS SUBPRODUCT         
			 , CASE WHEN cc.Queue = 'WHL' THEN 'WHL'
			        ELSE 'RTL'
					END                                                              AS RtlWhl
			 , cc.Queue
			 , REPLACE(ac.ExposureStatusCode,'074-','')                                  AS CASE_DENOUNCE_STATUS
			 , cc.Agency	--Code 
			 , ac.Bucket  
			, CASE WHEN ac.Currency = 'EUR' OR ac.Currency = 'ΕUR' OR ac.Currency is null THEN ac.DueAmount
               ELSE ac.DueAmount / ( SELECT max(fx.filled_fxrate)
                             FROM  `{env_prefix}-bi`.common.fx_rates fx
                             WHERE fx.OriginalCurrency = ac.Currency
                               AND fx.Date = ac.BusinessDate
                            )
               END AS CASE_DEBT_AMT 
			,CASE WHEN ac.Currency = 'EUR' OR ac.Currency = 'ΕUR' OR ac.Currency is null THEN ac.OutstandingTotalBalance
               ELSE ac.OutstandingTotalBalance / ( SELECT max(fx.filled_fxrate)
                             FROM  `{env_prefix}-bi`.common.fx_rates fx
                             WHERE fx.OriginalCurrency = ac.Currency
                               AND fx.Date = ac.BusinessDate
                            )
               END AS CASE_OUTSTANDING_AMT
    		 , CASE WHEN u.BucketCategoryWithMateriality <= 3           	THEN 'Risk Bucket <= 3'
		    	    WHEN ac.Bucket < 7                               		THEN 'Bucket < 7'
		        	WHEN REPLACE(ac.ExposurestatusCode,'074-','') = 'DENOUNCED' THEN 'DENOUNCED'			
			        WHEN cc.Queue LIKE '%WHL%'                            	THEN 'WHL'
					WHEN ec.ExceptionQueues IS NOT NULL                    	THEN 'Queue Exception'
				END                                                              AS ExceptionReason 
	FROM UnionFile                             u
    INNER JOIN Account  ac
	ON ac.ExposureKey = u.ExposureKey   
    LEFT JOIN `{env_prefix}-cdh`.primary.casecode cc   																		   
    ON cc.CaseCodeKey = ac.CaseCode AND cc.BusinessDate = ac.BusinessDate AND cc.PortfolioName='Core Retail'                   
    LEFT JOIN `{env_prefix}-cdh`.primary.product p 																			   
    ON p.ProductKey = ac.ProductKey AND p.BusinessDate = ac.BusinessDate AND p.SourceSystem = ac.SourceSystem				   
	LEFT JOIN  UnderLaw       l 
	ON l.CDI = ac.CDI 
	LEFT JOIN ExceptionQueues ec 																			
	ON LOWER(ec.ExceptionQueues) = LOWER(cc.Queue)                                   										
 	WHERE u.ECAAssetClass = 'Residential' AND ac.AssetSegment = 'ML' AND u.ExposureSegmentation > 3
	AND  aC.IsGuaranteeFlag = 'true'
  '''
)
MLPerimeter.createOrReplaceTempView("MLPerimeter")

In [0]:
SBPerimeter1 = spark.sql(
f'''		
 	SELECT ac.AssetSegment
		 , ac.ExposureKey    
	  	 , ac.AgreementKey
		 , ac.CDI
		 , ac.PartyTaxRegistrationNumber
		 , CASE WHEN u.BucketCategoryWithMateriality <= 3 THEN 'THR' 
			    WHEN u.BucketCategoryWithMateriality > 3  THEN 'NPL'
		   END                                                              AS NplThr
         , CASE WHEN l.CDI = ac.CDI THEN 'N3869' 
           ELSE 'No' END                                                    AS N3869
		 , p.SubProduct 												  	AS Product  
		 , CONCAT(p.BankingUnitCode,' - ',p.ProductDescription)  			AS SUBPRODUCT
		 , CASE WHEN cc.Queue = 'WHL' THEN 'WHL'
		   ELSE 'RTL'
		   END AS RtlWhl
		 , cc.Queue
		 , REPLACE(ac.ExposureStatusCode,'074-','')                              AS CASE_DENOUNCE_STATUS
		 , cc.Agency 								                         
		 , ac.Bucket  							                         
			,CASE WHEN ac.Currency = 'EUR' OR ac.Currency = 'ΕUR' OR ac.Currency is null THEN ac.DueAmount
               ELSE ac.DueAmount / ( SELECT max(fx.filled_fxrate)
                             FROM  `{env_prefix}-bi`.common.fx_rates fx
                             WHERE fx.OriginalCurrency = ac.Currency
                               AND fx.Date = ac.BusinessDate
                            )
               END AS CASE_DEBT_AMT
		 ,CASE WHEN ac.Currency = 'EUR' OR ac.Currency = 'ΕUR' OR ac.Currency is null THEN ac.OutstandingTotalBalance
               ELSE ac.OutstandingTotalBalance / ( SELECT max(fx.filled_fxrate)
                             FROM  `{env_prefix}-bi`.common.fx_rates fx
                             WHERE fx.OriginalCurrency = ac.Currency
                               AND fx.Date = ac.BusinessDate
                            )
               END AS CASE_OUTSTANDING_AMT
		 , CASE WHEN (a.ExposureKeyWithConfirmedDebts <> '' 
			        AND a.ExposureKeyWithConfirmedDebts <> ' - '
         			AND a.ExposureKeyWithConfirmedDebts <> '-') THEN 'Missing in Arxeio Apografis'
					--  (a.`Αριθμός Λογαριασμού Βεβαιωμένων οφειλών` <> '' 
			        -- AND a.`Αριθμός Λογαριασμού Βεβαιωμένων οφειλών` <> ' - '
         			-- AND a.`Αριθμός Λογαριασμού Βεβαιωμένων οφειλών` <> '-') THEN 'Missing in Arxeio Apografis' -- 20250901 FM
		            WHEN ec.ExceptionComments IS NULL						THEN 'Not in Exception Comments'
			        WHEN u.BucketCategoryWithMateriality <= 3             	THEN 'Risk Bucket <= 3'
			        WHEN ac.Bucket < 7                                    	THEN 'Bucket < 7'
			        WHEN cc.Queue LIKE '%WHL%'                            	THEN 'WHL'
					WHEN eq.ExceptionQueues IS NOT NULL 		            THEN 'Queue Exception' 
		   END                                                              AS ExceptionReason 
		FROM ArxeioApografis                          a
		INNER JOIN UnionFile                          u
		ON a.AccountNumber = u.ExposureKey
		INNER JOIN Account                           ac
		ON ac.ExposureKey = u.ExposureKey
    	LEFT JOIN `{env_prefix}-cdh`.primary.casecode cc   																		   
    	ON cc.CaseCodeKey = ac.CaseCode AND cc.BusinessDate = ac.BusinessDate AND cc.PortfolioName='Core Retail'                   
    	LEFT JOIN `{env_prefix}-cdh`.primary.product p 						                                    	               
  		ON p.ProductKey = ac.ProductKey AND p.BusinessDate = ac.BusinessDate AND p.SourceSystem = ac.SourceSystem	               
		--LEFT JOIN CoreSunolikiPerimetros sp 																					   --prepei na ftiaxtei o pinakas
		--ON a.AccountNumber = sp.AccountNumber 																				   --prepei na ftiaxtei o pinakas
	LEFT JOIN  UnderLaw       l 
	ON l.CDI = ac.CDI 
    	LEFT JOIN ExceptionQueues eq 																			
		ON LOWER(eq.ExceptionQueues) = LOWER(cc.Queue)                                   										
		LEFT JOIN ExceptionComments ec 
		ON ec.ExceptionComments = a.Comments --a.`ΣΧΟΛΙΑ` -- 20250901 FM
		WHERE u.ExposureSegmentation > 3 AND ac.AssetSegment = 'SB' 
  '''
)
SBPerimeter1.createOrReplaceTempView("SBPerimeter1")

In [0]:
SBPerimeter2 = spark.sql(
f'''
  SELECT ac.AssetSegment    
	     , ac.ExposureKey     
			 , ac.AgreementKey
			 , ac.CDI
			 , ac.PartyTaxRegistrationNumber
			 , CASE WHEN u.BucketCategoryWithMateriality <= 3 THEN 'THR' 
			        WHEN u.BucketCategoryWithMateriality > 3  THEN 'NPL'
					END                                                              		 AS NplThr
             , CASE WHEN l.CDI = ac.CDI THEN 'N3869' 
			        ELSE 'No' END                                                    AS N3869
			 , p.SubProduct 												  		 AS Product   
			 , CONCAT(p.BankingUnitCode,' - ',p.ProductDescription)                  AS SUBPRODUCT
			 , CASE WHEN cc.Queue LIKE '%WHL%' THEN 'WHL'
			        ELSE 'RTL'
					END                                                             		 AS RtlWhl
			 , cc.Queue 
			 , REPLACE(ac.ExposurestatusCode,'074-','')                                  AS CASE_DENOUNCE_STATUS
			 , cc.Agency --Code 
			 , ac.Bucket  
			, CASE WHEN ac.Currency = 'EUR' OR ac.Currency = 'ΕUR' OR ac.Currency is null THEN ac.DueAmount
               ELSE ac.DueAmount / ( SELECT max(fx.filled_fxrate)
                             FROM  `{env_prefix}-bi`.common.fx_rates fx
                             WHERE fx.OriginalCurrency = ac.Currency
                               AND fx.Date = ac.BusinessDate
                            )
               END AS CASE_DEBT_AMT
		 	 ,CASE WHEN ac.Currency = 'EUR' OR ac.Currency = 'ΕUR' OR ac.Currency is null THEN ac.OutstandingTotalBalance
               ELSE ac.OutstandingTotalBalance / ( SELECT max(fx.filled_fxrate)
                             FROM  `{env_prefix}-bi`.common.fx_rates fx
                             WHERE fx.OriginalCurrency = ac.Currency
                               AND fx.Date = ac.BusinessDate
                            )
               END AS CASE_OUTSTANDING_AMT 
			 , CASE WHEN u.BucketCategoryWithMateriality <= 3 THEN 'Risk Bucket <= 3'
			        WHEN ac.Bucket < 7                    	  THEN 'Bucket < 7'
			        WHEN cc.Queue = 'WHL'                     THEN 'WHL'
							WHEN eq.ExceptionQueues IS NOT NULL       THEN 'Queue Exception' 
					END                                                            			  AS ExceptionReason 
		FROM Covid19                                  c
		INNER JOIN UnionFile                          u
		ON c.AccountNumber = u.ExposureKey
		INNER JOIN Account                           ac
		ON ac.ExposureKey = u.ExposureKey
      	LEFT JOIN `{env_prefix}-cdh`.primary.casecode cc   																		  
        ON cc.CaseCodeKey = ac.CaseCode AND cc.BusinessDate = ac.BusinessDate AND cc.PortfolioName='Core Retail'                   
    	LEFT JOIN `{env_prefix}-cdh`.primary.product p 						                               						   
       	ON p.ProductKey = ac.ProductKey AND p.BusinessDate = ac.BusinessDate AND p.SourceSystem = ac.SourceSystem	               
		LEFT JOIN underlaw         l
		ON l.CDI = ac.CDI 
		LEFT JOIN ExceptionQueues eq    
		ON LOWER(eq.ExceptionQueues) = LOWER(cc.Queue)
		WHERE u.ExposureSegmentation > 3 AND ac.AssetSegment = 'SB'
  '''
)
SBPerimeter2.createOrReplaceTempView("SBPerimeter2")

In [0]:
Final = spark.sql(
f'''
    SELECT *
    FROM MLPerimeter

    UNION ALL

    SELECT *
    FROM SBPerimeter1

    UNION ALL

    SELECT *
    FROM SBPerimeter2

'''
)
Final.createOrReplaceTempView("Final")

# Final.display(5)

In [0]:
final_df = spark.sql(  
f'''
		SELECT CAST('{run_date}' AS DATE) 										AS BusinessDate
	          ,	 CAST(AssetSegment AS STRING)         							AS AssetSegment
			  ,	 CAST(final.ExposureKey AS STRING) 								AS ExposureKey
			  ,	 CAST(AgreementKey AS STRING)									AS AgreementKey
			  ,	 CAST(CDI AS STRING) 											AS CDI
			  ,	 CAST(PartyTaxRegistrationNumber AS STRING)						AS PartyTaxRegistrationNumber
			  ,	 CAST(NplThr AS STRING) 										AS NplThr
			  ,	 CAST(N3869 AS STRING) 											AS N3869
			  ,	 CAST(Product AS STRING) 										AS Product
			  ,	 CAST(SUBPRODUCT AS STRING) 									AS SUBPRODUCT
			  ,	 CAST(d.ContractCoCFinal AS STRING) 							AS FinalDEKStatus
			  ,	 CAST(d.AppealExclusionContract AS boolean) 					AS Enstasi
			  ,	 CAST(RtlWhl AS STRING) 										AS RtlWhl
			  ,	 CAST(Queue AS STRING)               							AS QueDesc
			  ,	 CAST(CASE_DENOUNCE_STATUS AS STRING) 							AS CASE_DENOUNCE_STATUS
			  ,	 CAST(Agency AS STRING) 										AS Agency
			  ,	 CAST(Bucket AS INT)          									AS CaseBucket
			  ,	 CAST(CASE_DEBT_AMT AS DECIMAL(19,2)) 							AS CASE_DEBT_AMT
			  ,  CAST(CASE_OUTSTANDING_AMT AS DECIMAL(19,2)) 					AS CASE_OUTSTANDING_AMT
			  ,	 CAST(ExceptionReason AS STRING) 								AS ExceptionReason
			  ,	 CAST(getdate() AS TIMESTAMP) AS InsertTimeStamp 
		FROM Final
  		LEFT JOIN DenouncementPool AS D ON Final.ExposureKey = D.ExposureKey 
  
  '''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_MLSBGovernmentGuaranteedBucket7Plus: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureKey"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
MLSBGovernmentGuaranteedBucket7Plus = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = {run_date!r}
'''
)
MLSBGovernmentGuaranteedBucket7Plus.createOrReplaceTempView("MLSBGovernmentGuaranteedBucket7Plus")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/MLSBGovernmentGuaranteedBucket7Plus/"
file_name = "MLSBGovernmentGuaranteedBucket7Plus_" +run_date+ ".csv"
MLSBGovernmentGuaranteedBucket7Plus.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)